[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/22_conv2d.ipynb)

# 🟠 Medium: 2D Convolution

Implement **2D convolution** from scratch.

### Signature
```python
def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    # x: (B, C_in, H, W), weight: (C_out, C_in, kH, kW)
    # Returns: (B, C_out, H_out, W_out)
```

### Rules
- Do NOT use `F.conv2d` or `nn.Conv2d`
- Support `stride` and `padding` parameters
- `F.pad` for zero-padding is allowed

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn.functional as F

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    """
    x:      (B, C_in, H, W)
    weight: (C_out, C_in, kH, kW)
    """

    # Allow both integers and (height, width) tuples.
    if isinstance(stride, int):
        stride_h = stride_w = stride
    else:
        stride_h, stride_w = stride

    if isinstance(padding, int):
        pad_h = pad_w = padding
    else:
        pad_h, pad_w = padding

    B, C_in, H, W = x.shape
    C_out, C_weight, kH, kW = weight.shape

    assert C_in == C_weight
    assert stride_h > 0 and stride_w > 0
    assert pad_h >= 0 and pad_w >= 0

    # 1. Zero padding
    x = F.pad(
        x,
        (pad_w, pad_w, pad_h, pad_h)
    )

    # 2. Calculate output dimensions
    H_padded, W_padded = x.shape[2:]

    H_out = (H_padded - kH) // stride_h + 1
    W_out = (W_padded - kW) // stride_w + 1

    assert H_out > 0 and W_out > 0

    # 3. Initialize output
    out = x.new_zeros((B, C_out, H_out, W_out))

    # 4. Slide the kernel over spatial positions
    for i in range(H_out):
        for j in range(W_out):
            h_start = i * stride_h
            w_start = j * stride_w

            # Extract a patch: (B, C_in, kH, kW)
            patch = x[
                :,
                :,
                h_start:h_start + kH,
                w_start:w_start + kW
            ]

            # Element-wise multiply and sum over
            # input channels and kernel dimensions.
            out[:, :, i, j] = (
                patch.unsqueeze(1) * weight.unsqueeze(0)
            ).sum(dim=(2, 3, 4))

    # 5. Add bias to each output channel
    if bias is not None:
        out = out + bias.view(1, C_out, 1, 1)

    return out



# def my_conv2d(x, weight, bias=None, stride=1, padding=0):
#     if padding > 0:
#         x = F.pad(x, [padding] * 4)
#     B, C_in, H, W = x.shape
#     C_out, _, kH, kW = weight.shape
#     H_out = (H - kH) // stride + 1
#     W_out = (W - kW) // stride + 1
#     patches = x.unfold(2, kH, stride).unfold(3, kW, stride)
#     out = torch.einsum('bihwjk,oijk->bohw', patches, weight)
#     if bias is not None:
#         out = out + bias.view(1, -1, 1, 1)
#     return out

In [4]:
# 🧪 Debug
x = torch.randn(1, 3, 8, 8)
w = torch.randn(16, 3, 3, 3)
print('Output:', my_conv2d(x, w).shape)
print('Match:', torch.allclose(my_conv2d(x, w), F.conv2d(x, w), atol=1e-4))

Output: torch.Size([1, 16, 6, 6])
Match: True


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('conv2d')


🧪 Testing: 2D Convolution (Medium)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (14.6ms)
  ✅ [2/5] Matches F.conv2d (241.0ms)
  ✅ [3/5] With padding (45.9ms)
  ✅ [4/5] With stride (24.7ms)
  ✅ [5/5] Gradient flow (61.3ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (387.5ms total)
  Progress saved. Run status() to see your dashboard.



In [15]:
# 예시 환경: 배치(B)=2, 입력채널(C_in)=3, 출력채널(C_out)=4, 돋보기크기(kH, kW)=3x3
patch = torch.ones((2, 3, 3, 3))       # (B, C_in, kH, kW)
weight = torch.ones((4, 3, 3, 3))      # (C_out, C_in, kH, kW)

print(f"1. 원본 patch 크기:  {patch.shape}")
print(f"2. 원본 weight 크기: {weight.shape}\n")

# 각각 빈 방(unsqueeze)을 끼워 넣기
patch_unsq = patch.unsqueeze(1)        # 1번 인덱스에 1 추가
weight_unsq = weight.unsqueeze(0)      # 0번 인덱스에 1 추가

print(f"3. 늘어난 patch 크기:  {patch_unsq.shape} -> (B, 1, C_in, kH, kW)")
print(f"4. 늘어난 weight 크기: {weight_unsq.shape} -> (1, C_out, C_in, kH, kW)\n")

# 두 텐서를 곱하면 PyTorch가 1인 부분을 서로의 크기에 맞춰 자동으로 늘려서 곱함
multiplied = patch_unsq * weight_unsq
print(f"5. 곱해진 최종 크기:   {multiplied.shape} -> (B, C_out, C_in, kH, kW)")

# 마지막으로 입력채널과 돋보기 공간(2, 3, 4번 인덱스)의 값들을 싹 다 더해버림
final_out = multiplied.sum(dim=(2, 3, 4))
print(f"6. Sum 이후 최종 크기: {final_out.shape} -> (B, C_out)")

1. 원본 patch 크기:  torch.Size([2, 3, 3, 3])
2. 원본 weight 크기: torch.Size([4, 3, 3, 3])

3. 늘어난 patch 크기:  torch.Size([2, 1, 3, 3, 3]) -> (B, 1, C_in, kH, kW)
4. 늘어난 weight 크기: torch.Size([1, 4, 3, 3, 3]) -> (1, C_out, C_in, kH, kW)

5. 곱해진 최종 크기:   torch.Size([2, 4, 3, 3, 3]) -> (B, C_out, C_in, kH, kW)
6. Sum 이후 최종 크기: torch.Size([2, 4]) -> (B, C_out)
